In [ ]:
# now lets pratice te RA using the API key form Google gemini ..


# SO we know tha RAG pipline right 
#  Documents -> Embedding -> Vectordatabase(here vetor indexing) it is same herre too 
# User Query -> query Embedding -> similarity search -> retrival from the documents -> prompt -> LLM ->  Answer -- but her we use and give ta prompt to gemni usig 
# gmani api key

# her eall tings same as befoe as we done before just the ptopmy is sended to the gemai

# 1. get the docyments chucks
# 2. make the embeddings and vectors and faiss indexing
# 3. get the user query and emdebbing it 
# 4. make the similariy using faiss and get the retrival context 
# 5. ssend retrica lcontex and  query to gemni and getthe answer

In [18]:
documents = [
    """
    RAG stands for Retrieval-Augmented Generation.
    It retrieves relevant external information and provides
    that information to a language model before generation.
    """,

    """
    Embeddings are numerical vector representations of text.
    They allow systems to compare the semantic similarity
    between pieces of text.
    """,

    """
    Chunking is the process of splitting large documents
    into smaller pieces so that relevant information can
    be retrieved efficiently.
    """,

    """
    A vector database stores vector representations and
    allows similarity search over those vectors.
    """
]

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(4, 384)


In [20]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)


4


In [45]:
query = "How does RAG retrieve information?"

In [46]:


query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

print(query_embedding)

[[-2.79781930e-02  1.02575876e-01 -1.61576774e-02  4.32331953e-03
  -6.10397942e-02  4.16215416e-03  4.18105237e-02 -3.75999417e-03
   2.99219671e-03 -6.47920975e-03  2.95955949e-02  9.57076997e-02
   2.94418409e-02 -7.95604065e-02 -8.21826607e-02  3.22837420e-02
   4.16922718e-02  1.19588509e-01 -7.06260353e-02 -4.25074436e-02
   6.78178214e-04  3.15209441e-02 -4.71500726e-03 -7.25936741e-02
   2.39517447e-02  7.49654099e-02 -9.81110036e-02 -8.75252206e-03
   7.48589784e-02 -6.87680617e-02  1.52158504e-03  4.64824215e-02
  -1.42133832e-02  2.47057490e-02 -4.79857661e-02  3.78168300e-02
  -4.05070260e-02  6.87658116e-02  1.57302953e-02 -1.02124165e-03
   8.35793675e-04  2.54156757e-02 -2.52725165e-02 -2.46472307e-03
   4.34511602e-02  5.69044426e-02 -4.71145939e-03  2.40529440e-02
   2.12320257e-02  8.91003944e-03 -8.15041289e-02  8.37899093e-03
   4.16475572e-02  9.01359841e-02 -4.85172346e-02 -2.27663107e-02
   4.19824384e-02 -5.21378145e-02 -4.71944734e-02  7.73457736e-02
  -4.72292

In [47]:
def retrieve(query, k=2):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    retrival_context = [documents[i] for i in indices[0]]
    

    return retrival_context

In [52]:
def build_prompt(query, retrieved_documents):

    context = "\n\n".join(retrieved_documents)

    prompt = f""" You are a helpful assistant.
            
            Answer the question using only the provided context.
            
            
            
            Context:
            {context}
            
            Question:
            {query}
            
            Answer:
            """

    return prompt

In [53]:
rag_prompt = build_prompt(query,retrieve(query,k=2))
print(rag_prompt)

 You are a helpful assistant.

            Answer the question using only the provided context.



            Context:
            
    RAG stands for Retrieval-Augmented Generation.
    It retrieves relevant external information and provides
    that information to a language model before generation.
    


    Chunking is the process of splitting large documents
    into smaller pieces so that relevant information can
    be retrieved efficiently.
    

            Question:
            How does RAG retrieve information?

            Answer:
            


In [ ]:
# lets swt up gemini ai and send te rag prompy to gemon ai using api key

In [ ]:
from google import genai
import os



client = genai.Client()


In [54]:
print("Sending structured RAG payload to Gemini 3.5 Flash...")

# Generate content via Gemini API using the context retrieved locally
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=rag_prompt
)

print("\n --- [Gemini RAG Output] ---")

print(response.text)

Sending structured RAG payload to Gemini 3.5 Flash...

 --- [Gemini RAG Output] ---
Based on the provided context, RAG retrieves relevant external information. This information is retrieved efficiently by splitting large documents into smaller pieces (a process called chunking) and is then provided to a language model before generation.
